In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Final Feature Layout (1662 dims/frame)
FACE:
468 × 3
= 1404

POSE:
33 × 4
(x,y,z,visibility)
= 132

LEFT HAND:
21 × 3
= 63

RIGHT HAND:
21 × 3
= 63

TOTAL:
1404 + 132 + 63 + 63
=
1662

Each video becomes:

(num_frames, 1662)

Example:

(182, 1662)

# SignVox Landmark Dataset

Generated using MediaPipe Holistic.

## Source

Input videos:

/content/S2S_DATASET_PHRASES/videos/

Output:

/content/extracted_landmarks/

Folder structure mirrors the original dataset.

---

## Frame Feature Layout

Each frame contains 1662 features.

### Face

468 landmarks

Each landmark:

x,y,z

468 × 3 = 1404

---

### Pose

33 landmarks

Each landmark:

x,y,z,visibility

33 × 4 = 132

---

### Left Hand

21 landmarks

x,y,z

21 × 3 = 63

---

### Right Hand

21 landmarks

x,y,z

21 × 3 = 63

---

## Total

1404 + 132 + 63 + 63

= 1662 features/frame

---

## NPY Shape

(num_frames, 1662)

Example:

(150, 1662)

---

## Missing Landmarks

If MediaPipe cannot detect a component,
its values are filled with zeros.

---

## Selfie Camera Correction

Videos were horizontally flipped
before landmark extraction to restore
real-world left/right orientation.

In [1]:
!pip uninstall -y mediapipe tensorflow protobuf
!pip install protobuf==4.25.3
!pip install mediapipe==0.10.14
!pip install -q opencv-python tqdm

Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Found existing installation: protobuf 5.29.5
Uninstalling protobuf-5.29.5:
  Successfully uninstalled protobuf-5.29.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 9.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-cloud-videointelligence 2.19.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
google-cloud-vision 3.14.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
a2a-sdk 0.3.26 requires protobuf>=5.29.5, but you have protobuf 4.25.3 which is i

In [2]:
import os
import gc
import json
import cv2
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

import mediapipe as mp

# =========================================================
# PATHS
# =========================================================

VIDEOS_ROOT = "/kaggle/input/datasets/basilsaji03/s2s-dataset-phrases/S2S_DATASET_PHRASES/content/S2S_DATASET_PHRASES/videos"

METADATA_CSV = "/kaggle/input/datasets/basilsaji03/s2s-dataset-phrases/metadata.csv"

OUTPUT_ROOT = "/kaggle/working/extracted_landmarks"

# =========================================================
# SETTINGS
# =========================================================

DEMIRROR_SELFIE_VIDEO = True

EXPECTED_FEATURE_DIM = 1662

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Config loaded.")

Config loaded.


In [3]:
metadata_df = pd.read_csv(METADATA_CSV)

print("Metadata rows:", len(metadata_df))
display(metadata_df.head())

Metadata rows: 418


,video_path,signer,phrase,signer_id,phrase_id,file_name
0,S2S_DATASET_PHRASES/videos/Signer 1 Albert/CAN...,S1,CAN I HELP YOU,0,0,VID_20260622_202637704.mp4
1,S2S_DATASET_PHRASES/videos/Signer 1 Albert/CAN...,S1,CAN I HELP YOU,0,0,VID_20260622_204006787.mp4
2,S2S_DATASET_PHRASES/videos/Signer 1 Albert/CAN...,S1,CAN I HELP YOU,0,0,VID_20260622_203719189.mp4
3,S2S_DATASET_PHRASES/videos/Signer 1 Albert/CAN...,S1,CAN I HELP YOU,0,0,VID_20260622_203604356.mp4
4,S2S_DATASET_PHRASES/videos/Signer 1 Albert/CAN...,S1,CAN I HELP YOU,0,0,VID_20260622_203053763.mp4


In [4]:
mp_holistic = mp.solutions.holistic

holistic = mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    refine_face_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

print("MediaPipe initialized.")

MediaPipe initialized.


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1782277554.961881     144 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782277555.000634     144 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782277555.001850     144 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782277555.002693     145 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782277555.003873     143 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782277555.020615     

In [6]:
def extract_frame_features(results):

    features = []

    # =====================================================
    # FACE
    # 468 x 3 = 1404
    # =====================================================

    if results.face_landmarks:
        for lm in results.face_landmarks.landmark:
            features.extend([
                lm.x,
                lm.y,
                lm.z
            ])
    else:
        features.extend([0.0] * (468 * 3))

    # =====================================================
    # POSE
    # 33 x 4 = 132
    # =====================================================

    if results.pose_landmarks:
        for lm in results.pose_landmarks.landmark:
            features.extend([
                lm.x,
                lm.y,
                lm.z,
                lm.visibility
            ])
    else:
        features.extend([0.0] * (33 * 4))

    # =====================================================
    # LEFT HAND
    # 21 x 3 = 63
    # =====================================================

    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            features.extend([
                lm.x,
                lm.y,
                lm.z
            ])
    else:
        features.extend([0.0] * (21 * 3))

    # =====================================================
    # RIGHT HAND
    # 21 x 3 = 63
    # =====================================================

    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            features.extend([
                lm.x,
                lm.y,
                lm.z
            ])
    else:
        features.extend([0.0] * (21 * 3))

    return np.asarray(features, dtype=np.float32)

In [7]:
def process_video(video_path):

    cap = cv2.VideoCapture(str(video_path))

    sequence = []

    while True:

        success, frame = cap.read()

        if not success:
            break

        if DEMIRROR_SELFIE_VIDEO:
            frame = cv2.flip(frame, 1)

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        results = holistic.process(rgb)

        frame_features = extract_frame_features(results)

        sequence.append(frame_features)

    cap.release()

    sequence = np.asarray(
        sequence,
        dtype=np.float32
    )

    if sequence.shape[0] == 0:
        raise RuntimeError(
            f"No frames extracted from {video_path}"
        )

    if sequence.shape[1] != EXPECTED_FEATURE_DIM:
        raise RuntimeError(
            f"Expected {EXPECTED_FEATURE_DIM} features "
            f"but got {sequence.shape[1]}"
        )

    return sequence

In [9]:
manifest_rows = []

total_frames = 0
processed = 0
failed = 0

for _, row in tqdm(
    metadata_df.iterrows(),
    total=len(metadata_df),
    desc="Processing Videos"
):

    try:

        # =====================================================
        # Example metadata path:
        #
        # S2S_DATASET_PHRASES/videos/
        # Signer 1 Albert/
        # CAN I HELP YOU/
        # VID_xxx.mp4
        # =====================================================

        relative_video_path = row["video_path"]

        metadata_path = Path(relative_video_path)

        # Remove:
        # S2S_DATASET_PHRASES/videos

        relative_part = Path(*metadata_path.parts[2:])

        # =====================================================
        # Build REAL video path
        # =====================================================

        absolute_video_path = (
            Path(VIDEOS_ROOT)
            / relative_part
        )

        # =====================================================
        # Safety check
        # =====================================================

        if not absolute_video_path.exists():

            raise FileNotFoundError(
                f"Video not found:\n{absolute_video_path}"
            )

        # =====================================================
        # Extract landmarks
        # =====================================================

        sequence = process_video(
            str(absolute_video_path)
        )

        # =====================================================
        # Output path
        #
        # extracted_landmarks/
        # └── Signer X/
        #     └── Phrase/
        #         └── VID_xxx.npz
        # =====================================================

        output_path = (
            Path(OUTPUT_ROOT)
            / relative_part
        ).with_suffix(".npz")

        output_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        # =====================================================
        # Save compressed NPZ
        # =====================================================

        np.savez_compressed(
            output_path,
            landmarks=sequence
        )

        # =====================================================
        # Manifest row
        # =====================================================

        manifest_rows.append({

            "video_path": str(relative_video_path),

            "landmark_path": str(
                output_path.relative_to(
                    OUTPUT_ROOT
                )
            ),

            "signer": row["signer"],
            "phrase": row["phrase"],

            "signer_id": row["signer_id"],
            "phrase_id": row["phrase_id"],

            "file_name": row["file_name"],

            "num_frames": int(
                sequence.shape[0]
            ),

            "feature_dim": int(
                sequence.shape[1]
            )
        })

        total_frames += sequence.shape[0]
        processed += 1

        # =====================================================
        # Memory cleanup
        # =====================================================

        del sequence
        gc.collect()

    except Exception as e:

        failed += 1

        print("\n" + "="*60)
        print("FAILED")
        print(relative_video_path)
        print(e)
        print("="*60)

print("\nExtraction Complete")
print(f"Processed : {processed}")
print(f"Failed    : {failed}")
print(f"Frames    : {total_frames}")

Processing Videos:   0%|          | 0/418 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[h264 @ 0x1637e4c0] Invalid NAL unit size (73256 > 12190).
[h264 @ 0x1637e4c0] missing picture in access unit with size 12194
[h264 @ 0x170f4f40] Invalid NAL unit size (73256 > 12190).
[h264 @ 0x170f4f40] Error splitting the input into NAL units.



Extraction Complete
Processed : 418
Failed    : 0
Frames    : 60083


In [10]:
manifest_df = pd.DataFrame(
    manifest_rows
)

manifest_path = os.path.join(
    OUTPUT_ROOT,
    "extraction_manifest.csv"
)

manifest_df.to_csv(
    manifest_path,
    index=False
)

print(manifest_path)
display(manifest_df.head())

/kaggle/working/extracted_landmarks/extraction_manifest.csv


,video_path,landmark_path,signer,phrase,signer_id,phrase_id,file_name,num_frames,feature_dim
0,S2S_DATASET_PHRASES/videos/Signer 1 Albert/CAN...,Signer 1 Albert/CAN I HELP YOU/VID_20260622_20...,S1,CAN I HELP YOU,0,0,VID_20260622_202637704.mp4,267,1662
1,S2S_DATASET_PHRASES/videos/Signer 1 Albert/CAN...,Signer 1 Albert/CAN I HELP YOU/VID_20260622_20...,S1,CAN I HELP YOU,0,0,VID_20260622_204006787.mp4,176,1662
2,S2S_DATASET_PHRASES/videos/Signer 1 Albert/CAN...,Signer 1 Albert/CAN I HELP YOU/VID_20260622_20...,S1,CAN I HELP YOU,0,0,VID_20260622_203719189.mp4,132,1662
3,S2S_DATASET_PHRASES/videos/Signer 1 Albert/CAN...,Signer 1 Albert/CAN I HELP YOU/VID_20260622_20...,S1,CAN I HELP YOU,0,0,VID_20260622_203604356.mp4,229,1662
4,S2S_DATASET_PHRASES/videos/Signer 1 Albert/CAN...,Signer 1 Albert/CAN I HELP YOU/VID_20260622_20...,S1,CAN I HELP YOU,0,0,VID_20260622_203053763.mp4,153,1662


In [11]:
summary = {

    "videos_processed": int(processed),

    "videos_failed": int(failed),

    "total_frames": int(total_frames),

    "feature_dimension": 1662,

    "face_features": 1404,

    "pose_features": 132,

    "left_hand_features": 63,

    "right_hand_features": 63
}

summary_path = os.path.join(
    OUTPUT_ROOT,
    "extraction_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(
        summary,
        f,
        indent=4
    )

print(summary_path)

/kaggle/working/extracted_landmarks/extraction_summary.json


In [12]:
readme = """
# S2S Landmark Dataset

Generated using MediaPipe Holistic.

Feature dimension per frame: 1662

Face:
468 x 3 = 1404

Pose:
33 x 4 = 132

Left Hand:
21 x 3 = 63

Right Hand:
21 x 3 = 63

Total:
1662

Storage:
Compressed NPZ

Shape:
(num_frames, 1662)

Videos were de-mirrored before extraction.
"""

with open(
    os.path.join(
        OUTPUT_ROOT,
        "README.md"
    ),
    "w"
) as f:
    f.write(readme)

print("README generated.")

README generated.


In [13]:
import random

sample = random.choice(manifest_rows)

sample_file = os.path.join(
    OUTPUT_ROOT,
    sample["landmark_path"]
)

arr = np.load(sample_file)["landmarks"]

print("Shape:", arr.shape)
print("Dtype:", arr.dtype)

Shape: (101, 1662)
Dtype: float32


In [14]:
import pandas as pd

manifest = pd.read_csv(
    "/kaggle/working/extracted_landmarks/extraction_manifest.csv"
)

print("Videos:", len(manifest))
print("Total Frames:", manifest["num_frames"].sum())
print("Average Frames:", manifest["num_frames"].mean())
print("Min Frames:", manifest["num_frames"].min())
print("Max Frames:", manifest["num_frames"].max())

Videos: 418
Total Frames: 60083
Average Frames: 143.73923444976077
Min Frames: 24
Max Frames: 360


In [15]:
import numpy as np
import pandas as pd
import os

manifest = pd.read_csv(
    "/kaggle/working/extracted_landmarks/extraction_manifest.csv"
)

sample = manifest.sample(10)

for _, row in sample.iterrows():

    path = os.path.join(
        "/kaggle/working/extracted_landmarks",
        row["landmark_path"]
    )

    arr = np.load(path)["landmarks"]

    print(
        row["file_name"],
        arr.shape
    )

VID_20260622_212453070.mp4 (138, 1662)
VID_20260622_202532.mp4 (145, 1662)
VID_20260622_185357.mp4 (133, 1662)
VID_20260622_191104.mp4 (88, 1662)
VID_20260622_222448558.mp4 (112, 1662)
VID_20260622_221045461.mp4 (135, 1662)
VID_20260623_150138.mp4 (95, 1662)
IMG_0913 (1).MOV (244, 1662)
VID_20260622_222218609.mp4 (148, 1662)
IMG_0951 (1).MOV (148, 1662)
